In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# RBFE many pairs 

End-to-end workflow for *two* relative binding free energy pair:

1. Load the BRD protein and three ligands, register them on the data platform
2. Run system prep in RBFE mode
3. Inspect the prepared system
4. Run RBFE FEP on the prepared system

## Setup

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Ligand,
    Protein,
    RBFE,
    RBFEParams,
    SystemPrep,
    PreparedSystem,
)
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()
client

## 1. Load structures and register on the data platform

We use the BRD4 example protein and two congeneric ligands from the bundled
dataset. `sync()` uploads files (when needed) and registers records on the
data platform.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()
protein.id

In [ ]:
ligands = []
for ligand_file in ["brd-2.sdf", "brd-3.sdf", "brd-4.sdf"]:
    ligand = Ligand.from_sdf(BRD_DATA_DIR / ligand_file)
    ligand.sync()
    ligands.append(ligand)
    

ligands

## 2. System prep (RBFE mode)

Pass both ligands to `SystemPrep` to prepare binding and solvation legs for
the pair. This runs synchronously via `deeporigin.system-prep`.

In [ ]:
def get_or_prepare_system(protein, ligand1, ligand2):

    try:
        systems = PreparedSystem.from_result(
            ligand1_id=ligand1.id,
            ligand2_id=ligand2.id,
            protein_id=protein.id,
        )
    except:
        systems = None

    if systems:
        return systems[0]

    return SystemPrep(
        protein=protein,
        ligand1=ligand1,
        ligand2=ligand2,
    ).run()


systems = [
    get_or_prepare_system(protein, lig1, lig2)
    for lig1, lig2 in zip(ligands[:-1], ligands[1:])
]

systems

## 3. Show the prepared systems

Visualize the solvated system PDB returned by system prep.

In [ ]:
systems[0].show()

In [ ]:
systems[1].show()

## 4. Run RBFE on the prepared system

Submit FEP only (`mode="rbfe"`) using the prepared binding/solvation XML paths.
We quote first, then confirm to start the job.

`test_run=1` shortens the simulation for exploration; use `test_run=0` for
production-quality results.

In [ ]:
rbfe = RBFE(
    prepared_systems=systems,
    params=RBFEParams(test_run=1),
)
rbfe

In [ ]:
rbfe.start(quote=True)
rbfe.estimate

In [ ]:
rbfe.confirm()

In [ ]:
task = await rbfe.watch()

## Results

In [ ]:
rbfe.get_results()

In [ ]:
files = client.files.list(f"tool-runs/{rbfe.id}")
for file in files:
    if "fep_results" in file:
        print(file)

In [ ]:
rbfe = RBFE.from_last_run()

In [ ]:
df = rbfe.get_user_logs()

In [ ]:
df